In [1]:
# =========================================================
# LOAD REQUIRED LIBRARIES
# =========================================================

import pandas as pd
import joblib


In [ ]:
# =========================================================
# LOAD SAVED FILES
# =========================================================

model = joblib.load('Models/aqi_model.pkl')

ohe = joblib.load('Models/city_encoder.pkl')

scaler = joblib.load('Models/scaler.pkl')

model_columns = joblib.load('Models/model_columns.pkl')


In [3]:
# =========================================================
# PREDICTION FUNCTION
# =========================================================

def prediction(input_data):

    # CONVERT INPUT INTO DATAFRAME

    new_data = pd.DataFrame(input_data)

    # =====================================================
    # PROCESS DATE COLUMN

    new_data['Date'] = pd.to_datetime(new_data['Date'])

    new_data['Year'] = new_data['Date'].dt.year

    new_data['Month'] = new_data['Date'].dt.month

    new_data['Weekday'] = new_data['Date'].dt.weekday

    # Drop Date column
    new_data.drop('Date', axis=1, inplace=True)

    # =====================================================
    # APPLY ONE HOT ENCODING
    

    city_encoded = ohe.transform(new_data[['City']])

    city_df = pd.DataFrame(

        city_encoded,

        columns=ohe.get_feature_names_out(['City'])

    )

    # Drop original city column
    new_data.drop('City', axis=1, inplace=True)

    # Add encoded columns
    new_data = pd.concat(

        [new_data, city_df],

        axis=1

    )

    # =====================================================
    # MATCH TRAINING COLUMNS
    

    new_data = new_data.reindex(

        columns=model_columns,

        fill_value=0

    )

    # =====================================================
    # APPLY SCALING

    new_data_scaled = scaler.transform(new_data)

    # =====================================================
    # PREDICT AQI

    predicted_aqi = model.predict(new_data_scaled)

    return predicted_aqi[0]

In [4]:
# =========================================================
# EXAMPLE INPUT
# =========================================================

input_data = {

    'City': ['Delhi'],

    'PM2.5': [180],

    'PM10': [320],

    'NO': [45],

    'NO2': [60],

    'NOx': [120],

    'NH3': [35],

    'CO': [2.1],

    'SO2': [18],

    'O3': [55],

    'Benzene': [12],

    'Toluene': [20],

    'Xylene': [5],

    'Date': ['2024-01-15']

}




In [6]:
# =========================================================
# FUNCTION CALL
# =========================================================

result = prediction(input_data)

print("Predicted AQI :", result)

Predicted AQI : Very Poor
